# Phase 1 — Lesson 18: Convex Optimization
> Faithful study notes built directly from the lesson source. Concepts, code, diagrams, and exercises.

**Source:** ai-engineering-from-scratch -> Phase 1 (Math Foundations), Lesson 18: Convex Optimization
**Type:** Build | **Languages:** Python | **Time:** ~90 minutes
**Prerequisites:** Phase 1, Lessons 04, 08



# What You'll Learn

---

## Learning Objectives

- Test whether a function is convex using the definition, second derivative, and Hessian criteria
- Implement Newton's method and compare its quadratic convergence against gradient descent
- Solve constrained optimization using Lagrange multipliers and interpret KKT conditions
- Explain why neural network loss landscapes are non-convex yet SGD still finds good solutions


# Convex Functions Have One Valley

---

A function is convex if its graph lies below chords:

```
f(tx + (1−t)y) ≤ t·f(x) + (1−t)·f(y)
```

Equivalently, the Hessian is positive semi-definite everywhere. Convex problems have **one global minimum**.


In [ ]:
import numpy as npimport matplotlib.pyplot as pltx = np.linspace(-3, 3, 400)plt.plot(x, x**2, label='f(x) = x² (convex)', color='#27ae60', lw=2)plt.plot(x, x**3 - 3*x, label='f(x) = x³ − 3x (non-convex)', color='#e74c3c', lw=2)plt.axhline(0, color='gray', lw=0.8); plt.axvline(0, color='gray', lw=0.8)plt.title('Convex vs Non-Convex Functions', fontsize=12, fontweight='bold')plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
import numpy as npdef hessian_2d(f, x, y, h=1e-5):    fxx = (f(x + h, y) - 2 * f(x, y) + f(x - h, y)) / h**2    fyy = (f(x, y + h) - 2 * f(x, y) + f(x, y - h)) / h**2    fxy = (f(x + h, y + h) - f(x + h, y - h) - f(x - h, y + h) + f(x - h, y - h)) / (4 * h**2)    return np.array([[fxx, fxy], [fxy, fyy]])def saddle(x, y): return x**2 - y**2def bowl(x, y): return x**2 + y**2H_saddle = hessian_2d(saddle, 0, 0)H_bowl = hessian_2d(bowl, 0, 0)print(f"Saddle Hessian:\n{H_saddle}")print(f"Eigenvalues: {np.linalg.eigvals(H_saddle)}")print(f"Bowl Hessian:\n{H_bowl}")print(f"Eigenvalues: {np.linalg.eigvals(H_bowl)}")

In [ ]:
import numpy as npimport matplotlib.pyplot as plt# Gradient descent vs Newton on quadratic bowl f(x,y) = 5x² + y²def f_quad(p): return 5 * p[0]**2 + p[1]**2def grad_quad(p): return np.array([10 * p[0], 2 * p[1]])def hess_quad(p): return np.array([[10, 0], [0, 2]])# Gradient descentp = np.array([10.0, 10.0])lr = 0.05path_gd = [p.copy()]for _ in range(200):    p = p - lr * grad_quad(p)    path_gd.append(p.copy())path_gd = np.array(path_gd)# Newton's methodp = np.array([10.0, 10.0])path_newton = [p.copy()]for _ in range(20):    g = grad_quad(p)    H = hess_quad(p)    p = p - np.linalg.solve(H, g)    path_newton.append(p.copy())path_newton = np.array(path_newton)print(f"Gradient descent: {len(path_gd)} steps to converge")print(f"Newton's method:  {len(path_newton)} steps to converge")# Plot contours with pathsx = np.linspace(-12, 12, 100)y = np.linspace(-6, 6, 100)X, Y = np.meshgrid(x, y)Z = 5 * X**2 + Y**2fig, axes = plt.subplots(1, 2, figsize=(12, 5))for ax, path, title in [(axes[0], path_gd, 'Gradient Descent'), (axes[1], path_newton, "Newton's Method")]:    ax.contour(X, Y, Z, levels=30, cmap='viridis', alpha=0.6)    ax.plot(path[:, 0], path[:, 1], 'ro-', markersize=4, lw=1.5)    ax.plot(path[0, 0], path[0, 1], 'ro', markersize=8, label='Start')    ax.plot(path[-1, 0], path[-1, 1], 'go', markersize=8, label='End')    ax.set_title(title, fontweight='bold')    ax.set_xlabel('x'); ax.set_ylabel('y')    ax.grid(True, alpha=0.3); ax.legend()plt.suptitle('Gradient Descent vs Newton on Elongated Bowl', fontsize=12, fontweight='bold')plt.tight_layout(); plt.show()

# Constrained Optimization: Lagrange Multipliers

---

Minimize f(x) subject to g(x) = 0. Form the Lagrangian:

```
L(x, λ) = f(x) + λ·g(x)
```

Example: minimize f(x, y) = x² + y² subject to x + y = 1.


In [ ]:
import numpy as np# Minimize f(x,y) = x² + y² subject to x + y = 1# Solution: x = y = 0.5x, y = 0.5, 0.5print(f"Constrained minimum: x = {x}, y = {y}")print(f"f(x,y) = {x**2 + y**2}")print(f"Check constraint: x + y = {x + y}")# Lagrange: dL/dx = 2x + λ = 0, dL/dy = 2y + λ = 0# So x = y = -λ/2, and x + y = 1 → x = y = 0.5, λ = -1lam = -1print(f"Lagrange multiplier λ = {lam}")

# Why Neural Networks Are Non-Convex

---

Neural network loss functions are wildly non-convex, yet SGD works because:
- Most local minima in high dimensions are near-optimal
- Saddle points vastly outnumber local minima
- SGD noise helps escape shallow minima


In [ ]:
import numpy as npimport matplotlib.pyplot as plt# Non-convex landscape with multiple local minimax = np.linspace(-2, 2, 400)y = np.sin(3 * x) + 0.5 * x**2plt.plot(x, y, color='#2c3e50', lw=2)plt.scatter([-1.2, 0.5, 1.3], [np.sin(3*-1.2) + 0.5*-1.2**2, np.sin(3*0.5) + 0.5*0.5**2, np.sin(3*1.3) + 0.5*1.3**2],            c=['red', 'green', 'red'], s=80, zorder=5)plt.annotate('Local min', xy=(-1.2, np.sin(3*-1.2) + 0.5*-1.2**2), xytext=(-1.5, 1.5),             arrowprops=dict(arrowstyle='->', color='red'))plt.annotate('Global min', xy=(0.5, np.sin(3*0.5) + 0.5*0.5**2), xytext=(0.2, 2.5),             arrowprops=dict(arrowstyle='->', color='green'))plt.title('Non-Convex Landscape: Multiple Local Minima', fontsize=12, fontweight='bold')plt.xlabel('x'); plt.ylabel('f(x)')plt.grid(True, alpha=0.3)plt.tight_layout(); plt.show()

# Quiz Review

---

Try answering these before reading the explanations:

**Q1:** Why is neural network loss non-convex?
> **Explanation:** Nonlinear activations create multiple local minima and saddle points.

**Q2:** What is Newton's method advantage?
> **Explanation:** Quadratic convergence near minimum — no learning rate to tune.

**Q3:** What is complementary slackness?
> **Explanation:** Either a constraint is active or its Lagrange multiplier is zero.



# Practice Exercises

---

Do these in order:

**Exercise 1:** Run gradient descent vs Newton on f(x, y) = 50x² + y² from (10, 10); count steps.

**Exercise 2:** Minimize f(x, y) = (x−3)² + (y−3)² subject to x + 2y = 4 using Lagrange multipliers.



# Key Terms

---

| Term | What people say | What it actually means |
|------|----------------|----------------------|
| Convex function | Line below chord | Every local minimum is the global minimum |
| Hessian | Matrix of 2nd derivatives | Positive semi-definite ⟺ convex |
| Newton's method | 2nd-order optimizer | Uses inverse Hessian — quadratic convergence |
| Lagrange multiplier | Constraint converter | Converts constrained to unconstrained problem |
| KKT conditions | Optimality conditions | Necessary conditions with inequality constraints |
| Complementary slackness | λ·g(x) = 0 | Either constraint active or multiplier zero |
| Duality | Primal ↔ dual | Same optimal value for convex problems |


# Further Reading

---

- [Boyd & Vandenberghe: Convex Optimization](https://web.stanford.edu/~boyd/cvxbook/)
- [Why Momentum Really Works (Distill)](https://distill.pub/2017/momentum/)


# Next Steps

---

When you're ready, continue to Lesson 19: Complex Numbers for AI.

Your `ai-engineering-from-scratch` learning directory is **completely untouched** — these notes live separately in this folder.
